# Numerical matching

In [475]:
import numpy as np
from scipy.optimize import minimize

def gam(alp, bet) :
    return (1+alp**2)/bet

def drift_matrix(L) :
    return np.array([[1, L],[0,1 ]])

def quad_matrix(f) :
    return np.array([[1, 0],[1/f,1 ]])

def quad_thick_matrix(L, k1) : 
    if k1 > 0 :
        return np.array([[np.cos(np.sqrt(k1)*L)               , 1/np.sqrt(k1)*np.sin(np.sqrt(k1)*L)], 
                         [-np.sqrt(k1)*np.sin(np.sqrt(k1)*L) , np.cos(np.sqrt(k1)*L)]]) 
    elif k1 < 0:
        k1 = abs(k1)
        return np.array([[np.cosh(np.sqrt(k1)*L)              , 1/np.sqrt(k1)*np.sinh(np.sqrt(k1)*L)], 
                         [np.sqrt(k1)*np.sinh(np.sqrt(k1)*L) , np.cosh(np.sqrt(k1)*L)]]) 

def twiss_propagation(m) :
    return np.array([[m[0,0]**2     ,   -2*m[0,0]*m[0,1]  ,     m[0,1]**2],
                     [-m[0,0]*m[1,0], 1+ 2*m[0,1]*m[1,0],  -m[0,1]*m[1,1]],
                     [m[1,0]**2     ,   -2*m[1,0]*m[1,1]  ,    m[1,1]**2]])

def matching_thin_transfer(d1, f1, d2, f2) :
    return quad_matrix(f2) @ drift_matrix(d2) @ quad_matrix(f1) @ drift_matrix(d1)

def matching_thick_transfer(d1, fd1, f1, d2, fd2, f2) :
    return quad_thick_matrix(fd1, 1/f2) @ drift_matrix(d2) @ quad_thick_matrix(fd1, 1/f1) @ drift_matrix(d1)

def matching_thin_twiss(d1,f1,d2,f2) :
    return twiss_propagation(matching_thin_transfer(d1, f1, d2, f2))

def matching_thick_twiss(d1, fd1, f1, d2, fd2, f2) :
    return twiss_propagation(matching_thick_transfer(d1, fd1, f1, d2, fd2, f2))

def match(m, m0 = [0,0], t0 = np.array([5,1.0]), t1 = np.array([1,0.0])) :
    t0 = np.array([t0[0], t0[1], gam(t0[1], t0[0])])
    t1 = np.array([t1[0], t1[1], gam(t1[1], t1[0])])

    def objective_thin(params) :
        t1m       = m(*(9,params[0],9,params[1])) @ t0
        sqr_diff  = (t1[0] - t1m[0])**2 
        sqr_diff += 100*(t1[1] - t1m[1])**2
        
        return sqr_diff

    def objective_thick(params) :
        t1m       = m(*(9,1.0,params[0],9,1.0,params[1])) @ t0
        sqr_diff  = (t1[0] - t1m[0])**2 
        sqr_diff += 100*(t1[1] - t1m[1])**2
        
        return sqr_diff
    
    result = minimize(objective_thin, 
                      x0=(m0[0],m0[1]), 
                      tol=0.001,
                      bounds=((-20,20),(-20,20)), 
                      options={'maxiter':10000}, 
                      method='Nelder-Mead') 

    print('t0',t0)
    print('t1',t1)
    print('m0',m0)
    print(result)
    print(m(9,result['x'][0],9,result['x'][1]) @ t0)
    #print(m(9,1.0,result['x'][0],9,1.0,result['x'][1]) @ t0)

In [476]:
match(matching_thin_twiss, m0=[-1, 1], 
      t0 = np.array([6.7,-5.44738202e-01]), 
      t1 = np.array([6.7,-5.44738202e-01]))

t0 [ 6.7        -0.5447382   0.19354324]
t1 [ 6.7        -0.5447382   0.19354324]
m0 [-1, 1]
       message: Optimization terminated successfully.
       success: True
        status: 0
           fun: 4.743069225468814e-07
             x: [-7.037e+00  6.150e+00]
           nit: 47
          nfev: 90
 final_simplex: (array([[-7.037e+00,  6.150e+00],
                       [-7.037e+00,  6.151e+00],
                       [-7.037e+00,  6.150e+00]]), array([ 4.743e-07,  1.001e-06,  3.255e-06]))
[ 6.70036084 -0.54479686  0.19354236]
